# Import packages and load data

In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from matplotlib import rc
from matplotlib import cm
from matplotlib import rc
from mpl_toolkits.axes_grid1.inset_locator import inset_axes  
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)  
from sklearn.preprocessing import StandardScaler  # or MinMaxScaler
import shap
from joblib import dump, load
xscaler = MinMaxScaler()

In [11]:
file_path = '../../pre-processing/likeness_data/poly_maccs.xlsx'
output1 = 'gisaxs_domain'
output2 = 'gisaxs_fwhm'
# import data frame for pre-processing
df = pd.read_excel(file_path)

# Data pre-processing

In [12]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def neighbor_noise_score(X, y, k=5):
    nn = NearestNeighbors(n_neighbors=k+1).fit(X)
    distances, indices = nn.kneighbors(X)
    
    # For each point, get mean absolute target diff with its neighbors
    diffs = []
    for i, neighbors in enumerate(indices[:, 1:]):  # exclude self
        diffs.append(np.mean(np.abs(y[i] - y[neighbors])))
    
    return np.mean(diffs)

In [13]:
# Drop any samples with missing data
df_clean = df.dropna()

# Assign input variables and target variable
inputs = df_clean.iloc[:, 1:-2].copy()
y1 = df_clean[f'{output1}']
y2 = df_clean[f'{output2}']

# Scale the inputs (fit only on training data to avoid data leakage)
scaler = StandardScaler()
inputs_scaled = scaler.fit_transform(inputs)

score1 = neighbor_noise_score(inputs_scaled, y1.reset_index(drop=True))
score2 = neighbor_noise_score(inputs_scaled, y2.reset_index(drop=True))

print(score1 / np.std(y1))  # lower = smoother = less noisy
print(score2 / np.std(y2))

0.14320932998057298
0.4225465598281128
